# MLflow Models: Model Signatures and Input Examples

In this notebook we will cover:
1. What is Model Signature
2. Automatic signature inference
3. Manual signature creation
4. Input examples
5. Input validation

## What is Model Signature?

**Model Signature** describes:
- Input data schema (column names and types)
- Output data schema
- Model parameters (optional)

**Why do we need it?**
- Input validation
- Documentation
- API generation
- Deployment safety

## 1. Import Libraries

In [1]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.types.schema import Schema, ColSpec
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import sys

sys.path.append('../src')
from data_loader import load_sample_data, get_sample_input
from model_inspector import inspect_model_structure

print(f"MLflow version: {mlflow.__version__}")

MLflow version: 3.9.0


## 2. Data Preparation

In [2]:
X_train, X_test, y_train, y_test, feature_names, target_names = load_sample_data('wine')

print(f"Training set: {X_train.shape}")
print(f"Features: {feature_names}")
print(f"Target classes: {len(target_names)}")

Training set: (142, 13)
Features: ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']
Target classes: 3


## 3. Model WITHOUT Signature

In [3]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("02_Model_Signatures")

with mlflow.start_run(run_name="no_signature") as run:
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)
    
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model"
    )
    
    run_id_no_sig = run.info.run_id
    print(f"Run ID: {run_id_no_sig}")

print("\nModel structure:")
inspect_model_structure(f"runs:/{run_id_no_sig}/model", verbose=True)

c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
2026/02/15 01:48:30 INFO mlflow.tracking.fluent: Experiment with name '02_Model_Signatures' does not exist. Creating a new experiment.
2026/02/15 01:48:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these forma

Run ID: 84293300220d4577835d2e5812aec2f0

Model structure:
MLflow Model Structure: runs:/84293300220d4577835d2e5812aec2f0/model

📁 Local path: C:\Users\PPOBER~1\AppData\Local\Temp\tmpxv56yo6z\model\

📄 Files in model:
  - MLmodel (777 bytes)
  - conda.yaml (263 bytes)
  - model.pkl (97,887 bytes)
  - python_env.yaml (121 bytes)
  - requirements.txt (123 bytes)

🏷️  Model flavors: ['python_function', 'sklearn']
⏰ Created: 2026-02-14 23:48:30.837174

🐍 Python version: python=3.11.6

📦 Requirements (8 packages):
  - mlflow==3.9.0
  - cloudpickle==3.1.2
  - numpy==2.4.2
  - pandas==2.3.3
  - psutil==7.2.2
  ... and 3 more



{'local_path': 'C:\\Users\\PPOBER~1\\AppData\\Local\\Temp\\tmpxv56yo6z\\model\\',
 'files': ['conda.yaml',
  'MLmodel',
  'model.pkl',
  'python_env.yaml',
  'requirements.txt'],
 'mlmodel': {'artifact_path': 'file:///c:/Learn/github/mlpops-basics/03_mlflow_models/notebooks/mlruns/128642946808582289/models/m-e509aef9511c4ac6a9ea98261b5e6654/artifacts',
  'flavors': {'python_function': {'env': {'conda': 'conda.yaml',
     'virtualenv': 'python_env.yaml'},
    'loader_module': 'mlflow.sklearn',
    'model_path': 'model.pkl',
    'predict_fn': 'predict',
    'python_version': '3.11.6'},
   'sklearn': {'code': None,
    'pickled_model': 'model.pkl',
    'serialization_format': 'cloudpickle',
    'sklearn_version': '1.8.0',
    'skops_trusted_types': None}},
  'mlflow_version': '3.9.0',
  'model_id': 'm-e509aef9511c4ac6a9ea98261b5e6654',
  'model_size_bytes': 97887,
  'model_uuid': 'm-e509aef9511c4ac6a9ea98261b5e6654',
  'prompts': None,
  'run_id': '84293300220d4577835d2e5812aec2f0',
  'ut

## 4. Model WITH Automatic Signature Inference

In [4]:
with mlflow.start_run(run_name="auto_signature") as run:
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)
    
    X_sample = pd.DataFrame(X_train[:5], columns=feature_names)
    y_pred = model.predict(X_sample)
    
    signature = infer_signature(X_sample, y_pred)
    print("Inferred signature:")
    print(signature)
    
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature
    )
    
    run_id_auto = run.info.run_id
    print(f"\nRun ID: {run_id_auto}")

c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
2026/02/15 01:48:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


Inferred signature:
inputs: 
  ['alcohol': double (required), 'malic_acid': double (required), 'ash': double (required), 'alcalinity_of_ash': double (required), 'magnesium': double (required), 'total_phenols': double (required), 'flavanoids': double (required), 'nonflavanoid_phenols': double (required), 'proanthocyanins': double (required), 'color_intensity': double (required), 'hue': double (required), 'od280/od315_of_diluted_wines': double (required), 'proline': double (required)]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None


Run ID: 98213d913b7f4aad9153ca507c92e12c


## 5. Model WITH Input Example

In [5]:
with mlflow.start_run(run_name="with_input_example") as run:
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)
    
    X_sample = pd.DataFrame(X_train[:5], columns=feature_names)
    y_pred = model.predict(X_sample)
    
    signature = infer_signature(X_sample, y_pred)
    
    input_example = X_sample.iloc[:1]
    
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=input_example
    )
    
    run_id_with_example = run.info.run_id
    print(f"Run ID: {run_id_with_example}")
    print("\nInput example:")
    print(input_example)

c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
2026/02/15 01:48:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


Run ID: 7cc110485cbe4d329d5357cd8ace6295

Input example:
   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    13.28        1.64  2.84               15.5      110.0            2.6   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        2.68                  0.34             1.36              4.6  1.09   

   od280/od315_of_diluted_wines  proline  
0                          2.78    880.0  


c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


### Inspect model with input example

In [6]:
print("Model structure:")
inspect_model_structure(f"runs:/{run_id_with_example}/model", verbose=True)

Model structure:
MLflow Model Structure: runs:/7cc110485cbe4d329d5357cd8ace6295/model

📁 Local path: C:\Users\PPOBER~1\AppData\Local\Temp\tmpnk_o6y_m\model\

📄 Files in model:
  - MLmodel (1,980 bytes)
  - conda.yaml (263 bytes)
  - input_example.json (313 bytes)
  - model.pkl (97,887 bytes)
  - python_env.yaml (121 bytes)
  - requirements.txt (123 bytes)
  - serving_input_example.json (593 bytes)

🏷️  Model flavors: ['python_function', 'sklearn']
⏰ Created: 2026-02-14 23:48:51.372874

🐍 Python version: python=3.11.6

📦 Requirements (8 packages):
  - mlflow==3.9.0
  - cloudpickle==3.1.2
  - numpy==2.4.2
  - pandas==2.3.3
  - psutil==7.2.2
  ... and 3 more



{'local_path': 'C:\\Users\\PPOBER~1\\AppData\\Local\\Temp\\tmpnk_o6y_m\\model\\',
 'files': ['conda.yaml',
  'input_example.json',
  'MLmodel',
  'model.pkl',
  'python_env.yaml',
  'requirements.txt',
  'serving_input_example.json'],
 'mlmodel': {'artifact_path': 'file:///c:/Learn/github/mlpops-basics/03_mlflow_models/notebooks/mlruns/128642946808582289/models/m-8cf4d687a67b47f4ba9b68d24f5b4d29/artifacts',
  'flavors': {'python_function': {'env': {'conda': 'conda.yaml',
     'virtualenv': 'python_env.yaml'},
    'loader_module': 'mlflow.sklearn',
    'model_path': 'model.pkl',
    'predict_fn': 'predict',
    'python_version': '3.11.6'},
   'sklearn': {'code': None,
    'pickled_model': 'model.pkl',
    'serialization_format': 'cloudpickle',
    'sklearn_version': '1.8.0',
    'skops_trusted_types': None}},
  'is_signature_from_type_hint': False,
  'mlflow_version': '3.9.0',
  'model_id': 'm-8cf4d687a67b47f4ba9b68d24f5b4d29',
  'model_size_bytes': 98793,
  'model_uuid': 'm-8cf4d687a67

## 6. Manual Signature Creation

In [7]:
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

input_schema = Schema([
    ColSpec("double", "alcohol"),
    ColSpec("double", "malic_acid"),
    ColSpec("double", "ash"),
    ColSpec("double", "alcalinity_of_ash"),
    ColSpec("double", "magnesium"),
    ColSpec("double", "total_phenols"),
    ColSpec("double", "flavanoids"),
    ColSpec("double", "nonflavanoid_phenols"),
    ColSpec("double", "proanthocyanins"),
    ColSpec("double", "color_intensity"),
    ColSpec("double", "hue"),
    ColSpec("double", "od280/od315_of_diluted_wines"),
    ColSpec("double", "proline"),
])

output_schema = Schema([ColSpec("long")])

manual_signature = ModelSignature(inputs=input_schema, outputs=output_schema)
print("Manual signature:")
print(manual_signature)

Manual signature:
inputs: 
  ['alcohol': double (required), 'malic_acid': double (required), 'ash': double (required), 'alcalinity_of_ash': double (required), 'magnesium': double (required), 'total_phenols': double (required), 'flavanoids': double (required), 'nonflavanoid_phenols': double (required), 'proanthocyanins': double (required), 'color_intensity': double (required), 'hue': double (required), 'od280/od315_of_diluted_wines': double (required), 'proline': double (required)]
outputs: 
  [long (required)]
params: 
  None



In [8]:
with mlflow.start_run(run_name="manual_signature") as run:
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)
    
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=manual_signature
    )
    
    run_id_manual = run.info.run_id
    print(f"\nRun ID: {run_id_manual}")

2026/02/15 01:49:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)



Run ID: bfdfee5765cc419b8b500816b39370ae


## 7. Input Validation

### Load model with signature

In [9]:
model_with_sig = mlflow.pyfunc.load_model(f"runs:/{run_id_auto}/model")

X_test_df = pd.DataFrame(X_test[:5], columns=feature_names)
predictions = model_with_sig.predict(X_test_df)
print("Valid predictions:")
print(predictions)

Valid predictions:
[0 2 0 1 1]


c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


### Test with incorrect data

In [10]:
wrong_data = pd.DataFrame({
    'wrong_column': [1.0, 2.0, 3.0]
})

try:
    predictions = model_with_sig.predict(wrong_data)
except Exception as e:
    print(f"Error (expected): {type(e).__name__}")
    print(f"Message: {str(e)}")

Error (expected): MlflowException
Message: Failed to enforce schema of data '   wrong_column
0           1.0
1           2.0
2           3.0' with schema '['alcohol': double (required), 'malic_acid': double (required), 'ash': double (required), 'alcalinity_of_ash': double (required), 'magnesium': double (required), 'total_phenols': double (required), 'flavanoids': double (required), 'nonflavanoid_phenols': double (required), 'proanthocyanins': double (required), 'color_intensity': double (required), 'hue': double (required), 'od280/od315_of_diluted_wines': double (required), 'proline': double (required)]'. Error: Model is missing inputs ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']. Note that there were extra inputs: ['wrong_column'].


### Test with wrong number of columns

In [11]:
wrong_columns = pd.DataFrame(X_test[:5, :5], columns=feature_names[:5])

try:
    predictions = model_with_sig.predict(wrong_columns)
except Exception as e:
    print(f"Error (expected): {type(e).__name__}")
    print(f"Message: {str(e)}")

Error (expected): MlflowException
Message: Failed to enforce schema of data '   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium
0    14.10        2.16  2.30               18.0      105.0
1    12.51        1.24  2.25               17.5       85.0
2    13.87        1.90  2.80               19.4      107.0
3    11.56        2.05  3.23               28.5      119.0
4    13.67        1.25  1.92               18.0       94.0' with schema '['alcohol': double (required), 'malic_acid': double (required), 'ash': double (required), 'alcalinity_of_ash': double (required), 'magnesium': double (required), 'total_phenols': double (required), 'flavanoids': double (required), 'nonflavanoid_phenols': double (required), 'proanthocyanins': double (required), 'color_intensity': double (required), 'hue': double (required), 'od280/od315_of_diluted_wines': double (required), 'proline': double (required)]'. Error: Model is missing inputs ['total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proantho

## 8. Comparing Models With and Without Signature

In [13]:
from model_inspector import compare_models

print("Comparing models:")
compare_models(
    model_uri1=f"runs:/{run_id_no_sig}/model",
    model_uri2=f"runs:/{run_id_auto}/model"
)

Comparing models:
Model Comparison

🏷️  Model 1 flavors: ['python_function', 'sklearn']
🏷️  Model 2 flavors: ['python_function', 'sklearn']
 Same flavors: True

📄 Same files: True


{'same_flavors': True,
 'flavors_1': ['python_function', 'sklearn'],
 'flavors_2': ['python_function', 'sklearn'],
 'same_files': True,
 'files_only_in_1': [],
 'files_only_in_2': []}

## Conclusions

In this notebook we learned:

1.  What is Model Signature and why it's important
2.  Automatic signature inference using `infer_signature()`
3.  Manual signature creation with `Schema` and `ColSpec`
4.  Adding input examples
5.  Input validation at inference time
6.  Comparing models with and without signatures

### Best Practices:
-  Always add signatures to production models
-  Include input examples for API documentation
-  Use automatic inference when possible
-  Manual signatures for custom validation

### Next Steps:
- **Notebook 03**: Different Model Flavors